# Domain Infrastructure Correlation

Identify shared infrastructure pivots across synthetic domain-registration and hosting observations.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Rank infrastructure clusters for analyst review without treating shared hosting as proof of common ownership.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
domain_count = 72
infrastructure = pd.DataFrame({
    "domain": [f"example-{index:03d}.test" for index in range(domain_count)],
    "registrar": rng.choice(["Registrar-A", "Registrar-B", "Registrar-C"], domain_count, p=[0.45, 0.35, 0.20]),
    "nameserver": rng.choice([f"ns{index}.synthetic.net" for index in range(1, 9)], domain_count),
    "asn": rng.choice([64510, 64511, 64512, 64513, 64514], domain_count),
    "domain_age_days": rng.integers(2, 1800, domain_count),
    "rare_tld": rng.binomial(1, 0.18, domain_count),
    "privacy_service": rng.binomial(1, 0.42, domain_count),
})
infrastructure["observed_alert"] = (
    (infrastructure["domain_age_days"] < 45)
    & (infrastructure["rare_tld"] == 1)
).astype(int)
print(infrastructure.head(6).to_string(index=False))


          domain   registrar        nameserver   asn  domain_age_days  rare_tld  privacy_service  observed_alert
example-000.test Registrar-A ns8.synthetic.net 64513              281         1                0               0
example-001.test Registrar-B ns2.synthetic.net 64510             1723         0                1               0
example-002.test Registrar-B ns8.synthetic.net 64510              175         0                0               0
example-003.test Registrar-B ns5.synthetic.net 64514             1256         0                0               0
example-004.test Registrar-B ns8.synthetic.net 64513              149         0                0               0
example-005.test Registrar-A ns6.synthetic.net 64513              587         0                1               0


### 2. Analyze and rank the observations


In [3]:
cluster_summary = (
    infrastructure.groupby(["nameserver", "asn"], as_index=False)
    .agg(domains=("domain", "count"), alerts=("observed_alert", "sum"), median_age=("domain_age_days", "median"))
)
cluster_summary["review_score"] = (
    0.45 * np.minimum(cluster_summary["domains"] / 8, 1)
    + 0.40 * np.minimum(cluster_summary["alerts"] / 3, 1)
    + 0.15 * (cluster_summary["median_age"] < 90)
).round(3)
ranked_clusters = cluster_summary.sort_values(["review_score", "domains"], ascending=False)
print(ranked_clusters.head(8).to_string(index=False))


       nameserver   asn  domains  alerts  median_age  review_score
ns5.synthetic.net 64510        1       1         9.0         0.340
ns1.synthetic.net 64512        5       0      1512.0         0.281
ns7.synthetic.net 64514        5       0       919.0         0.281
ns8.synthetic.net 64513        5       0       586.0         0.281
ns2.synthetic.net 64513        4       0       732.0         0.225
ns7.synthetic.net 64510        4       0       185.5         0.225
ns2.synthetic.net 64510        3       0       951.0         0.169
ns4.synthetic.net 64514        3       0       879.0         0.169


## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert infrastructure["domain"].is_unique
assert ranked_clusters["review_score"].between(0, 1).all()
assert ranked_clusters["domains"].sum() == len(infrastructure)
print("Checks passed; shared infrastructure remains a review lead, not an attribution claim.")


Checks passed; shared infrastructure remains a review lead, not an attribution claim.


## Next Steps

- Add passive-DNS timestamps and source provenance.
- Require an independent signal before escalating a cluster.
